# Deployment Decision: Labeling Scheme, Architecture, and Ensemble

**What this notebook settles.** Every previous measurement of the deployment model
was made under one labeling scheme and, in most cases, a single seed. Two questions
were therefore unanswerable:

1. **Is the midpoint labeling inflating the numbers?** Training windows were labeled
   `y = labels[mid]` — the centre of the window — while the window extends 30 beats
   past that point. A model predicting the centre using the whole window is using
   information from after the moment it is labelling. Deployment needs the *end* of
   the window (`y = labels[e-1]`): predict now, from what has already happened.
2. **Which configuration should ship?** MS-CGCA scored below the old BiLSTM in the
   one controlled comparison run so far, and XGBoost alone sat between them — all
   within a single-seed noise band, so none of it was decisive.

```
4 configurations x 2 labeling schemes x 5 seeds = 40 LOSO runs
```

**Runtime is 9-12 h on a T4** — longer than one Kaggle session. Every run is cached
to `RESULTS_PATH` as it completes; re-running this notebook **skips finished work and
resumes**. Expect to run it across two or three sessions. Nothing is lost between them
as long as the output file is preserved (`/kaggle/working` persists in a saved version;
download it if unsure).

---

## Design decisions that make this a fair test

**All configurations are evaluated on identical windows.** The previous ablation
compared a three-way run on 9,650 windows against two-way runs on 12,026 — not
like-for-like, as it noted itself. Here `strat_calib` is computed for *every*
configuration and every model is scored on `eval_local` only. The two-way and
XGBoost-only configurations simply never use the calibration slice. Same denominator,
same windows, every row.

**Ensemble weights are selected nested.** For each held-out subject, weights are chosen
on the other fourteen subjects only. Weights chosen on the same pooled predictions being
reported inflate the result — this was measured at roughly +0.007 F1 previously and is
excluded here by construction.

**The circadian feature follows the label.** `bi` is taken at whatever index the label
is taken from. Under endpoint labeling the time-of-day feature is computed at `e-1`,
not at the midpoint — otherwise the feature vector would still carry a timestamp from
after the labelled moment.

**Both metrics are reported.** Macro-F1 and quadratic kappa moved in opposite directions
across seeds in earlier runs (F1 stable, kappa swinging 0.026). Neither is assumed
steadier; both are reported with a standard deviation across seeds.


## 1. Setup

In [1]:
!pip install neurokit2 xgboost -q

   ━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━ 389.1/688.9 kB 11.8 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 11.9 MB/s eta 0:00:00


In [2]:
import os, pickle, warnings, json, time, itertools
import numpy as np, pandas as pd
from scipy.signal import welch
try:
    from scipy.integrate import trapezoid as TRAPZ
except ImportError:
    from scipy.integrate import trapz as TRAPZ
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import cohen_kappa_score
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow.keras import layers, callbacks, Model
from tensorflow.keras.losses import Loss
import neurokit2 as nk
warnings.filterwarnings('ignore')

DATA_PATH = '/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD'
SAVE = '/kaggle/working'; os.makedirs(SAVE, exist_ok=True)
RESULTS_PATH = f'{SAVE}/deployment_decision.json'

SUBJECT_IDS = [2,3,4,5,6,7,8,9,10,11,13,14,15,16,17]
CLASS_NAMES = ['relaxed','mild','moderate','high']; NCLS = 4

WINDOW = 60            # settled: 60 beats beat 120 for both architectures
STEP   = 5
EWMA_HALFLIVES = {'fast':60, 'medium':300, 'slow':1800}
POPULATION_RR_MS = 780.0
ROLL_WINDOW = 20
ZSCORE_HALFLIFE = 300

SEEDS      = [42, 7, 13, 2024, 99]
LABELINGS  = ['midpoint', 'endpoint']
CONFIGS    = ['xgb_only', 'mscgca_2way', 'bilstm_2way', 'mscgca_3way']

print("TF", tf.__version__, "GPU", len(tf.config.list_physical_devices('GPU')) > 0)
print(f"planned runs: {len(CONFIGS)} configs x {len(LABELINGS)} labelings x {len(SEEDS)} seeds "
      f"= {len(CONFIGS)*len(LABELINGS)*len(SEEDS)}")

TF 2.20.0 GPU True
planned runs: 4 configs x 2 labelings x 5 seeds = 40


## 2. Pipeline

Preprocessing, causal replacements and feature builders are transcribed from
`notebook-newmodel.ipynb` cells 2-3. Any divergence here would make this notebook
measure something other than the deployment pipeline.

In [3]:
def load_subject(sid):
    with open(f"{DATA_PATH}/S{sid}/S{sid}.pkl", 'rb') as f:
        data = pickle.load(f, encoding='latin1')
    return data['signal']['chest'], data['signal']['wrist']['TEMP'].flatten(), data['label'].flatten()

def extract_rr_from_ecg(ecg, fs=700):
    ecg = nk.ecg_clean(ecg.flatten(), sampling_rate=fs)
    _, info = nk.ecg_peaks(ecg, sampling_rate=fs); rp = info['ECG_R_Peaks']
    return np.diff(rp)*(1000.0/fs), (rp[:-1]+rp[1:])/2.0/fs, rp

def clean_rr(rr, ts):
    rr = rr.copy().astype(float); rr[(rr <= 300) | (rr >= 2000)] = np.nan
    for i in range(1, len(rr)):
        if not np.isnan(rr[i-1]) and not np.isnan(rr[i]):
            if abs(rr[i]-rr[i-1])/rr[i-1] > 0.20: rr[i] = np.nan
    m = np.isnan(rr)
    if m.any(): rr[m] = np.interp(np.where(m)[0], np.where(~m)[0], rr[~m])
    return rr, ts.copy()

def align_temp(wt, rp, fe=700, ft=4.0):
    tap = np.interp(rp/fe, np.arange(len(wt))/ft, wt); return (tap[:-1]+tap[1:])/2.0

def labels_to_rr(labels, rp):
    out = []
    for i in range(len(rp)-1):
        seg = labels[rp[i]:rp[i+1]]; v = seg[seg > 0]
        out.append(0 if len(v) == 0 else np.bincount(v).argmax())
    return np.array(out)

# ---- causal replacements (past-only by construction) ----
def ewma_causal(x, halflife):
    a = 1 - np.exp(np.log(0.5)/max(halflife, 1))
    o = np.empty(len(x), dtype=float); state = float(POPULATION_RR_MS)
    for i in range(len(x)):
        state = a*x[i] + (1-a)*state; o[i] = state
    return o

def causal_zscore(x, halflife=ZSCORE_HALFLIFE):
    a = 1 - np.exp(np.log(0.5)/max(halflife, 1))
    mu = np.empty(len(x)); sd = np.empty(len(x))
    m = float(x[0]) if len(x) else 0.0; v = 1.0
    for i in range(len(x)):
        d = x[i]-m; m = m + a*d; v = (1-a)*(v + a*d*d)
        mu[i] = m; sd[i] = np.sqrt(max(v, 1e-8))
    return (x-mu)/(sd+1e-8)

def roll_rmssd_causal(x, w=ROLL_WINDOW):
    o = np.zeros(len(x))
    for i in range(len(x)):
        seg = x[max(0, i-w+1):i+1]
        o[i] = np.sqrt(np.mean(np.diff(seg)**2)) if len(seg) > 1 else 0.0
    return o

def roll_sdnn_causal(x, w=ROLL_WINDOW):
    o = np.zeros(len(x))
    for i in range(len(x)):
        seg = x[max(0, i-w+1):i+1]
        o[i] = np.std(seg) if len(seg) > 1 else 0.0
    return o

def hrv_features(w, fs=4.0):
    rr, diff = np.array(w), np.diff(w)
    mean_rr = np.mean(rr); sdnn = np.std(rr); rmssd = np.sqrt(np.mean(diff**2))
    pnn50 = np.sum(np.abs(diff) > 50)/len(diff)*100; cv = sdnn/mean_rr
    t = np.cumsum(rr)/1000.0; u = np.interp(np.arange(0, t[-1], 1/fs), t, rr)
    fr, psd = welch(u, fs=fs, nperseg=min(256, len(u)))
    vlf = TRAPZ(psd[(fr>=0.003)&(fr<0.04)]); lf = TRAPZ(psd[(fr>=0.04)&(fr<0.15)])
    hf  = TRAPZ(psd[(fr>=0.15)&(fr<0.40)])
    lf_hf = lf/(hf+1e-8); lf_nu = lf/(lf+hf+1e-8)
    sd1 = np.sqrt(0.5)*np.std(diff); sd2 = np.sqrt(max(2*sdnn**2 - 0.5*np.var(diff), 0))
    return np.array([mean_rr, sdnn, rmssd, pnn50, cv, vlf, lf, hf, lf_hf, lf_nu,
                     sd1, sd2, sd1/(sd2+1e-8)])

def resid_features(rw):
    r = np.array(rw)
    return np.array([np.mean(r), np.std(r), np.max(np.abs(r)),
                     np.polyfit(np.arange(len(r)), r, 1)[0], np.sum(r**2)/len(r)])

def circ_features(ts):
    t, hour = ts % 86400, (ts % 86400)/3600.0
    cort = 0.6*np.exp(-0.5*((hour-8)/1.5)**2) + 0.3*np.exp(-0.5*((hour-15)/1.5)**2)
    return np.array([np.sin(2*np.pi*t/86400), np.cos(2*np.pi*t/86400),
                     np.sin(2*np.pi*t/5400), np.cos(2*np.pi*t/5400), cort])

def circ7(ts):
    t = ts % 86400; hour = t/3600.0
    return np.array([np.sin(2*np.pi*t/86400), np.cos(2*np.pi*t/86400),
                     np.sin(2*np.pi*t/5400), np.cos(2*np.pi*t/5400),
                     0.6*np.exp(-0.5*((hour-8)/1.5)**2) + 0.3*np.exp(-0.5*((hour-15)/1.5)**2),
                     np.sin(2*np.pi*(hour-23)/24), np.cos(2*np.pi*(hour-23)/24)])

print("pipeline defined")

pipeline defined


### Load subjects once, then build one feature set per labeling scheme

The only difference between the two builds is the index the label and the time-of-day
feature are taken from: `mid` versus `e-1`. Everything else — the windows themselves,
the sequence channels, the HRV and residual features — is identical.

In [4]:
raw = {}
for sid in SUBJECT_IDS:
    try:
        chest, wt, labels = load_subject(sid); ecg = chest['ECG'].flatten()
        rr, ts, rp = extract_rr_from_ecg(ecg); temp = align_temp(wt, rp)
        rr, ts = clean_rr(rr, ts)
        rl = labels_to_rr(labels, rp); keep = rl > 0
        rrk, tk, tsk, lk = rr[keep], temp[keep], ts[keep], rl[keep]
        new = np.zeros(len(lk), dtype=int); si = np.where(lk == 2)[0]
        if len(si) > 0:
            srr = rrk[si]; loc = []
            for i in range(len(srr)):
                w = srr[max(0, i-15):i+15]; dd = np.diff(w)
                loc.append(np.sqrt(np.mean(dd**2)) if len(dd) > 0 else 50)
            loc = np.array(loc); p33, p66 = np.percentile(loc, 33), np.percentile(loc, 66)
            for i, idx in enumerate(si):
                new[idx] = (1 if loc[i] >= p66 else 2 if loc[i] >= p33 else 3)
        raw[sid] = dict(rr=rrk, temp=tk, ts=tsk, lab=new)
    except Exception as e:
        print("FAIL", sid, e)
print(len(raw), "subjects loaded"); assert len(raw) == 15


def build(labeling):
    # labeling: 'midpoint' -> label at window centre; 'endpoint' -> label at last beat.
    Xs, Xc, Xx, y, g = [], [], [], [], []
    for sid, d in raw.items():
        rr, temp, ts, labels = d['rr'], d['temp'], d['ts'], d['lab']
        base = {k: ewma_causal(rr, hl) for k, hl in EWMA_HALFLIVES.items()}
        res_med  = rr - base['medium']
        temp_res = temp - ewma_causal(temp, EWMA_HALFLIVES['medium'])
        rn  = causal_zscore(rr); rm = roll_rmssd_causal(rn); sd = roll_sdnn_causal(rn)
        hr  = 60000/(rr+1e-8); rrn = causal_zscore(res_med)
        tn  = causal_zscore(temp); trn = causal_zscore(temp_res)
        for s in range(0, len(rr)-WINDOW, STEP):
            e = s + WINDOW
            li = (s + WINDOW//2) if labeling == 'midpoint' else (e - 1)   # label index
            bi = min(li, len(ts)-1)                                        # follows the label
            seq = np.stack([rn[s:e], rm[s:e], sd[s:e], hr[s:e],
                            rrn[s:e], tn[s:e], trn[s:e]], axis=-1)
            try:
                xf = np.concatenate([hrv_features(rr[s:e]), resid_features(res_med[s:e]),
                                     np.array([base['fast'][e-1], base['slow'][e-1]]),
                                     circ_features(ts[bi])])
            except Exception:
                continue
            Xs.append(seq); Xc.append(circ7(ts[bi])); Xx.append(xf)
            y.append(labels[li]); g.append(sid)
    return (np.array(Xs, np.float32), np.array(Xc, np.float32), np.array(Xx),
            np.array(y, np.int32), np.array(g, np.int32))

DATA = {}
for lb in LABELINGS:
    t0 = time.time(); DATA[lb] = build(lb)
    Xs, Xc, Xx, y, g = DATA[lb]
    print(f"{lb:9s} seq {Xs.shape}  xgb {Xx.shape}  classes {np.bincount(y, minlength=4)}  ({time.time()-t0:.0f}s)")

15 subjects loaded


midpoint  seq (12026, 60, 7)  xgb (12026, 25)  classes [8774 1101 1080 1071]  (12s)


endpoint  seq (12026, 60, 7)  xgb (12026, 25)  classes [8775 1099 1085 1067]  (12s)


## 3. Architectures

`build_ms_cgca` is transcribed from `notebook-newmodel.ipynb`; `build_bilstm` is the
previous architecture, included so the two can be compared at the same window size and
under the same labeling.

Note on causality: MS-CGCA is causal per timestep (causal padding, unidirectional LSTM),
but its final `GlobalAveragePooling1D` still aggregates across the whole window. Under
**endpoint** labeling that is not a leak — every position in the window precedes the
labelled beat. Under **midpoint** labeling it is one, for both architectures. This is
precisely what the labeling comparison measures.

In [5]:
class SparseFocalLoss(Loss):
    def __init__(self, gamma=2.0):
        super().__init__(); self.gamma = gamma
    def call(self, yt, yp):
        yt = tf.cast(yt, tf.int32)
        ce = tf.keras.losses.sparse_categorical_crossentropy(yt, yp)
        pt = tf.reduce_sum(tf.one_hot(yt, 4)*yp, axis=-1)
        return tf.pow(1.0-pt, self.gamma)*ce

def build_ms_cgca(window=WINDOW, nch=7, ncirc=7, ncls=4):
    si = layers.Input(shape=(window, nch), name='sequence')
    ci = layers.Input(shape=(ncirc,), name='circadian')
    c1 = layers.Conv1D(32, 3, padding='causal', activation='relu', dilation_rate=1)(si)
    c2 = layers.Conv1D(32, 3, padding='causal', activation='relu', dilation_rate=2)(si)
    c4 = layers.Conv1D(32, 3, padding='causal', activation='relu', dilation_rate=4)(si)
    x = layers.Concatenate()([c1, c2, c4])
    x = layers.BatchNormalization()(x); x = layers.MaxPooling1D(2)(x)
    x = layers.LSTM(128, return_sequences=True)(x)
    x = layers.Dropout(0.4)(x)
    cp = layers.Dense(128, activation='relu')(ci)
    cp = layers.RepeatVector(window//2)(cp)
    attn = layers.Attention()([cp, x])
    x = layers.GlobalAveragePooling1D()(attn)
    cf = layers.Dense(32, activation='relu')(ci)
    m = layers.Concatenate()([x, cf])
    o = layers.Dense(64, activation='relu')(m); o = layers.Dropout(0.4)(o)
    return Model([si, ci], layers.Dense(ncls, activation='softmax')(o), name='MS_CGCA')

def build_bilstm(window=WINDOW, nch=7, ncirc=7, ncls=4):
    si = layers.Input(shape=(window, nch), name='sequence')
    ci = layers.Input(shape=(ncirc,), name='circadian')
    x = layers.Conv1D(64, 7, padding='same', activation='relu')(si)
    x = layers.BatchNormalization()(x); x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(128, 5, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x); x = layers.MaxPooling1D(2)(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
    x = layers.Dropout(0.4)(x); a = layers.Attention()([x, x])
    x = layers.GlobalAveragePooling1D()(a)
    c = layers.Dense(32, activation='relu')(ci); c = layers.Dense(16, activation='relu')(c)
    x = layers.Concatenate()([x, c]); x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    return Model([si, ci], layers.Dense(ncls, activation='softmax')(x), name='BiLSTM')

def make_xgb(seed):
    return XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
                         colsample_bytree=0.8, objective='multi:softprob', num_class=4,
                         eval_metric='mlogloss', random_state=seed, n_jobs=-1)

def freeze_extractor(m):
    for l in m.layers:
        ln = l.__class__.__name__.lower()
        l.trainable = not any(k in ln for k in ['conv','lstm','batchnorm','attention','pooling'])
    return m

def strat_calib(sub_idx, y, frac=0.2, minpc=5):
    lab = y[sub_idx]; ntot = int(len(sub_idx)*frac); calib = []
    rng = np.random.RandomState(42)          # fixed: the split must not vary with seed
    for c in np.unique(lab):
        pos = sub_idx[lab == c]
        take = min(max(minpc, ntot//len(np.unique(lab))), len(pos))
        calib.extend(rng.choice(pos, take, replace=False))
    calib = np.array(sorted(calib))
    ev = np.array([i for i in sub_idx if i not in set(calib)])
    return calib, ev

print("architectures defined")

architectures defined


## 4. Runner

One function per (config, labeling, seed). Every configuration computes `strat_calib`
and scores on `eval_local`, so all four are compared on identical windows regardless of
whether they use the calibration slice. Per-fold predictions are stored so the ensemble
weight search can be run nested afterwards.

In [6]:
def macro_f1(yt, yp, K=4):
    f = []
    for c in range(K):
        tp = np.sum((yp==c)&(yt==c)); fp = np.sum((yp==c)&(yt!=c)); fn = np.sum((yp!=c)&(yt==c))
        f.append(0.0 if tp == 0 else 2*tp/(2*tp+fp+fn))
    return float(np.mean(f))

GRID = [(wf, round(wx*(1-wf), 4), round((1-wx)*(1-wf), 4))
        for wf in [0.0, 0.1, 0.2, 0.3] for wx in np.arange(0.3, 0.71, 0.1)]

def blend(f, w):
    wf, wx, wc = w
    p = wx*np.array(f['p_xgb'])
    if f['p_cnn'] is not None: p = p + wc*np.array(f['p_cnn'])
    if f['p_ft']  is not None: p = p + wf*np.array(f['p_ft'])
    return p

def run_one(config, labeling, seed):
    np.random.seed(seed); tf.random.set_seed(seed)
    Xs, Xc, Xx, y, g = DATA[labeling]
    three_way = (config == 'mscgca_3way')
    use_net   = (config != 'xgb_only')
    arch      = 'bilstm' if config == 'bilstm_2way' else 'mscgca'

    folds = []
    for tr, te in LeaveOneGroupOut().split(Xx, y, g):
        calib, ev = strat_calib(te, y)          # computed for ALL configs -> same eval set

        sc = StandardScaler().fit(Xx[tr])
        xgb = make_xgb(seed)
        xgb.fit(sc.transform(Xx[tr]), y[tr],
                sample_weight=compute_sample_weight('balanced', y[tr]), verbose=False)
        p_xgb = xgb.predict_proba(sc.transform(Xx[ev]))

        p_cnn = p_ft = None
        if use_net:
            cw = compute_class_weight('balanced', classes=np.unique(y[tr]), y=y[tr])
            net = (build_bilstm if arch == 'bilstm' else build_ms_cgca)()
            net.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss=SparseFocalLoss(2.0))
            cb = [callbacks.EarlyStopping(monitor='val_loss', patience=15,
                                          restore_best_weights=True, verbose=0),
                  callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                              patience=7, verbose=0)]
            net.fit([Xs[tr], Xc[tr]], y[tr], validation_split=0.15, epochs=120,
                    batch_size=32, class_weight=dict(enumerate(cw)), callbacks=cb, verbose=0)
            p_cnn = net.predict([Xs[ev], Xc[ev]], verbose=0)

            if three_way:
                ft = (build_bilstm if arch == 'bilstm' else build_ms_cgca)()
                ft.set_weights(net.get_weights()); ft = freeze_extractor(ft)
                ft.compile(optimizer=tf.keras.optimizers.Adam(5e-5), loss=SparseFocalLoss(2.0))
                if len(np.unique(y[calib])) >= 2:
                    cwc = compute_class_weight('balanced', classes=np.unique(y[calib]), y=y[calib])
                    cwdc = dict(zip(np.unique(y[calib]), cwc))
                else:
                    cwdc = None
                ft.fit([Xs[calib], Xc[calib]], y[calib], epochs=15, batch_size=8,
                       class_weight=cwdc, verbose=0)
                p_ft = ft.predict([Xs[ev], Xc[ev]], verbose=0)

        folds.append(dict(y=y[ev].tolist(), p_xgb=p_xgb.tolist(),
                          p_cnn=None if p_cnn is None else p_cnn.tolist(),
                          p_ft=None if p_ft is None else p_ft.tolist()))
        tf.keras.backend.clear_session()

    # ---- nested weight selection (skipped when there is nothing to weight) ----
    if config == 'xgb_only':
        preds = [np.array(f['p_xgb']).argmax(1) for f in folds]
    else:
        preds = []
        for i, fS in enumerate(folds):
            inner = [f for j, f in enumerate(folds) if j != i]
            innerY = np.concatenate([f['y'] for f in inner])
            def inner_f1(w):
                return macro_f1(innerY, np.concatenate([blend(f, w).argmax(1) for f in inner]))
            w_star = max(GRID, key=inner_f1)
            preds.append(blend(fS, w_star).argmax(1))

    yt = np.concatenate([f['y'] for f in folds]); yp = np.concatenate(preds)
    per_subj = [macro_f1(np.array(f['y']), p) for f, p in zip(folds, preds)]
    return dict(config=config, labeling=labeling, seed=seed,
                n_eval=int(len(yt)),
                f1=macro_f1(yt, yp),
                kappa=float(cohen_kappa_score(yt, yp, weights='quadratic')),
                acc=float(np.mean(yt == yp)),
                severe=float(np.mean(np.abs(yt-yp) >= 2)),
                within1=float(np.mean(np.abs(yt-yp) <= 1)),
                per_subject_f1=per_subj)

print("runner ready")

runner ready


## 5. Run everything (resumable)

Each completed run is written to disk immediately. If the session dies, re-run this cell
— finished runs are skipped. `xgb_only` runs first so that a full labeling comparison
exists early even if the networks are still pending.

In [7]:
store = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}
print(f"resuming with {len(store)} completed runs\n")

todo = [(c, lb, sd) for c in CONFIGS for lb in LABELINGS for sd in SEEDS]
t_start = time.time()
for config, labeling, seed in todo:
    key = f"{config}|{labeling}|{seed}"
    if key in store:
        print(f"{key:38s} cached  F1={store[key]['f1']:.4f}")
        continue
    t0 = time.time()
    print(f"{key:38s} running...", end=' ', flush=True)
    store[key] = run_one(config, labeling, seed)
    json.dump(store, open(RESULTS_PATH, 'w'))
    print(f"F1={store[key]['f1']:.4f}  kappa={store[key]['kappa']:.4f}  ({(time.time()-t0)/60:.1f} min)")

print(f"\ntotal this session: {(time.time()-t_start)/60:.1f} min")
print(f"completed overall : {len(store)}/{len(todo)}")

resuming with 0 completed runs

xgb_only|midpoint|42                   running... 

F1=0.6519  kappa=0.7966  (0.8 min)
xgb_only|midpoint|7                    running... 

F1=0.6568  kappa=0.7965  (0.8 min)
xgb_only|midpoint|13                   running... 

F1=0.6543  kappa=0.8022  (0.8 min)
xgb_only|midpoint|2024                 running... 

F1=0.6518  kappa=0.7879  (0.8 min)
xgb_only|midpoint|99                   running... 

...

F1=0.6692  kappa=0.8135  (22.2 min)
mscgca_3way|endpoint|42                running... 

F1=0.5869  kappa=0.7645  (23.7 min)
mscgca_3way|endpoint|7                 running... 

F1=0.6036  kappa=0.7864  (23.6 min)
mscgca_3way|endpoint|13                running... 

F1=0.6118  kappa=0.8043  (22.9 min)
mscgca_3way|endpoint|2024              running... 

F1=0.5941  kappa=0.7539  (23.7 min)
mscgca_3way|endpoint|99                running... 

## 6. Results

In [ ]:
store = json.load(open(RESULTS_PATH))
rows = list(store.values())
if not rows:
    print("no runs yet"); raise SystemExit

df = pd.DataFrame([{k: v for k, v in r.items() if k != 'per_subject_f1'} for r in rows])

print("="*94)
print("MEAN +/- SD ACROSS SEEDS")
print("="*94)
agg = (df.groupby(['labeling','config'])
         .agg(n=('seed','count'),
              f1_mean=('f1','mean'), f1_sd=('f1','std'),
              k_mean=('kappa','mean'), k_sd=('kappa','std'),
              acc=('acc','mean'), severe=('severe','mean'), n_eval=('n_eval','first'))
         .reset_index())
print(f"{'labeling':<11}{'config':<15}{'n':>3}{'macro-F1':>18}{'kappa':>18}{'acc':>8}{'severe':>8}{'n_eval':>8}")
print("-"*94)
for _, r in agg.iterrows():
    sd_f1 = 0.0 if pd.isna(r.f1_sd) else r.f1_sd
    sd_k  = 0.0 if pd.isna(r.k_sd)  else r.k_sd
    print(f"{r.labeling:<11}{r['config']:<15}{int(r.n):>3}"
          f"{r.f1_mean:>11.4f} +/-{sd_f1:<5.4f}"
          f"{r.k_mean:>11.4f} +/-{sd_k:<5.4f}"
          f"{r.acc:>8.4f}{r.severe:>8.4f}{int(r.n_eval):>8}")
print("="*94)

In [ ]:
print("="*72); print("A.  LABELING INFLATION  (midpoint - endpoint, same config & seed)"); print("="*72)
for config in CONFIGS:
    pairs = []
    for sd in SEEDS:
        a = store.get(f"{config}|midpoint|{sd}"); b = store.get(f"{config}|endpoint|{sd}")
        if a and b: pairs.append((a['f1']-b['f1'], a['kappa']-b['kappa']))
    if not pairs:
        print(f"{config:<15} (incomplete)"); continue
    df1 = np.mean([p[0] for p in pairs]); dk = np.mean([p[1] for p in pairs])
    print(f"{config:<15} n={len(pairs)}  dF1={df1:+.4f}   dkappa={dk:+.4f}")
print("\nA positive dF1 means the midpoint scheme scored higher, i.e. the previously")
print("reported numbers were inflated by that much, and the endpoint column is")
print("the honest deployment figure.\n")

print("="*72); print("B.  CONFIGURATION RANKING  (endpoint labeling only)"); print("="*72)
end_rows = [r for r in rows if r['labeling'] == 'endpoint']
if end_rows:
    e = pd.DataFrame([{k: v for k, v in r.items() if k != 'per_subject_f1'} for r in end_rows])
    rank = e.groupby('config').agg(f1=('f1','mean'), sd=('f1','std'),
                                   kappa=('kappa','mean'), ksd=('kappa','std')).sort_values('f1', ascending=False)
    print(rank.round(4).to_string())
    print("\nDifferences smaller than the seed standard deviation are not real differences.")
else:
    print("(no endpoint runs yet)")

In [ ]:
# Paired per-subject test between the top two endpoint configs
from scipy.stats import wilcoxon
end = [r for r in rows if r['labeling'] == 'endpoint']
if end:
    means = {}
    for c in CONFIGS:
        v = [r['per_subject_f1'] for r in end if r['config'] == c]
        if v: means[c] = np.mean(np.array(v), axis=0)      # avg across seeds, per subject
    if len(means) >= 2:
        order = sorted(means, key=lambda c: means[c].mean(), reverse=True)
        a, b = order[0], order[1]
        stat, p = wilcoxon(means[a], means[b])
        d = (means[a]-means[b]).mean()/((means[a]-means[b]).std(ddof=1)+1e-12)
        print("="*72); print(f"C.  {a}  vs  {b}   (endpoint, per-subject, seed-averaged)"); print("="*72)
        print(f"  {a:<15} mean per-subject F1 = {means[a].mean():.4f}")
        print(f"  {b:<15} mean per-subject F1 = {means[b].mean():.4f}")
        print(f"  mean difference = {d*0 + (means[a]-means[b]).mean():+.4f}")
        print(f"  Wilcoxon p = {p:.4f}   d = {d:+.2f}")
        print("  " + ("SIGNIFICANT" if p < 0.05 else "not significant at alpha=0.05"))
        print("\n  If this is not significant, prefer the simpler model: fewer dependencies,")
        print("  no TensorFlow at inference, one artifact to version.")

## 7. How to read this

**A — labeling.** The endpoint column is the deployable number. Whatever the midpoint
column reported previously, only the endpoint figure describes a model predicting the
present from the past. If the gap is large, earlier deployment numbers were optimistic
and `docs/ARCHITECTURE.md` needs updating with the endpoint figures.

**B — configuration.** Read the standard deviations before the means. A gap between two
configurations that is smaller than either one's seed-to-seed spread is not evidence of
a difference. If `xgb_only` is within noise of the networks, ship it: no TensorFlow at
inference, one artifact instead of three, and none of the personalisation machinery.

**C — the three-way ensemble.** Now compared on the same windows as everything else, so
the previous objection (9,650 versus 12,026) no longer applies. It still trains its third
member on the held-out subject's own calibration slice, so a gain here is partly the
value of personalisation, not of ensembling. Treat it as an upper bound on what per-user
calibration buys, and note that a deployed version needs that calibration to happen live.

**What this does not settle.** Window size (60 beats was already decided, on both
architectures). Anything about the mechanism findings in the ICAC paper — those use a
different pipeline and are unaffected. Whether a shorter window than 60 would do better,
which was never tested.

**After this runs.** Update `docs/ARCHITECTURE.md` with the endpoint figures, retrain the
chosen configuration on all 15 subjects, and export `model_config.json` plus the fitted
scaler together in the same run so their provenance is unambiguous.